In [60]:
import numpy as np
import torch
import torchvision

from torch import cuda
from torch.nn.functional import nll_loss, log_softmax
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from utils.progress import progressbar as pgb
import matplotlib.pyplot as plt

In [61]:
import pandas as pd

## Exploring the data

In [39]:
fpath = "data/titanic.csv"
df = pd.read_csv(fpath)

In [46]:
df.columns.to_list()

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

### Columns
source: https://www.kaggle.com/datasets/yasserh/titanic-dataset

"Pclass", "Age", "SibSp", "Parch", „Fare“, „Sex“, „Embarked“

- `PassengerId`: Passenger ID
- `Survived`*: Whether Survived or not: 0 = No, 1 = Yes
- `Pclass`*: Ticket class: 1 = 1st, 2 = 2nd, 3 = 3rd
- `Name`: Name of the Passenger
- `Sex`*: Gender
- `Age`*: Age in Years
- `SibSp`*: No. of siblings / spouses aboard the Titanic
- `Parch`*: No. of parents / children aboard the Titanic
- `Ticket`: Ticket number
- `Fare`*: Passenger fare
- `Cabin`: Cabin ID of passenger
- `Embarked`*: Port where passenger got on ship (`'S'`, `'C'`, `'Q'`, `nan`)

In [89]:
df['Embarked'].unique()

<StringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype: str

### Dropped columns

Generally discrete or (almost) unique values for each passenger.
Some may have a pattern, but would require extensive knowledge on the respective values to be interpreted.

- PassengerId: Passenger ID
- Name: Name of the Passenger
- Ticket: Ticket number
- Cabin: Cabin ID of passenger

#### PassengerId

Unique id for each passenger. No knowledge of whether consecutive numbers or patterns have any meaning.

#### Name

We have no reason to expect the name of a passenger to have any influence. Names have too many unique values.

#### Ticket

Often numbers, but also has different patterns. Therefore, not a number. Too many unique values to be interpreted as category.
Not useful unless interpreted with domain knowledge.

#### Cabin

Assigned cabin ID. Cabin IDs are not explained. Irregular pattern. Lots of `nan` (687/891)

In [90]:
df['Cabin']

0       NaN
1       C85
2       NaN
3      C123
4       NaN
       ... 
886     NaN
887     B42
888     NaN
889    C148
890     NaN
Name: Cabin, Length: 891, dtype: str

In [72]:
import re

In [95]:
# lots of NaN for Cabins
df['Cabin'].isna().sum()

np.int64(687)

In [88]:
def is_unusual(row):
    v = row['Cabin']
    return not (v is np.nan or re.fullmatch('^[A-Z]\\d{1,3}', v))

# Unusual cabins
df[df.apply(is_unusual, axis=1)]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.00,3,2,19950,263.0000,C23 C25 C27,S
75,76,0,3,"Moen, Mr. Sigurd Hansen",male,25.00,0,0,348123,7.6500,F G73,S
88,89,1,1,"Fortune, Miss. Mabel Helen",female,23.00,3,2,19950,263.0000,C23 C25 C27,S
97,98,1,1,"Greenfield, Mr. William Bertram",male,23.00,0,1,PC 17759,63.3583,D10 D12,C
118,119,0,1,"Baxter, Mr. Quigg Edmond",male,24.00,0,1,PC 17558,247.5208,B58 B60,C
128,129,1,3,"Peter, Miss. Anna",female,NaN,1,1,2668,22.3583,F E69,C
292,293,0,2,"Levy, Mr. Rene Jacques",male,36.00,0,0,SC/Paris 2163,12.8750,D,C
297,298,0,1,"Allison, Miss. Helen Loraine",female,2.00,1,2,113781,151.5500,C22 C26,S
299,300,1,1,"Baxter, Mrs. James (Helene DeLaudeniere Chaput)",female,50.00,0,1,PC 17558,247.5208,B58 B60,C
305,306,1,1,"Allison, Master. Hudson Trevor",male,0.92,1,2,113781,151.5500,C22 C26,S
